In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
# Importing Libraries
import os
import tensorflow as tf
import pandas as pd
import numpy as np
import librosa
import librosa.display
from tqdm import tqdm

In [28]:
data_path = "/content/drive/MyDrive/SER" # the dataset we are using is the RAVDESS.

In [29]:
# emotion mapping

emotion_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

In [30]:
# features extraction
def extract_features(data, sr):

  # MFCC
  mfcc = np.mean(librosa.feature.mfcc(y=data, sr=sr, n_mfcc=40).T, axis = 0)

  # Chroma

  Chrma = np.mean(librosa.feature.chroma_stft(y=data, sr=sr).T, axis=0)

  # Mel Spectrogram(Texture)

  mel = np.mean(librosa.feature.melspectrogram(y=data, sr=sr).T, axis=0)

  # combining the features of all

  return np.hstack([mfcc, Chrma, mel])

In [36]:
# audio augmentation

def noise(data):

  noise_amp = (
      0.035 * np.random.uniform() * np.amax(data)
  )

  return data + noise_amp * np.random.normal(size=data.shape[0])
def stretch(data, rate=0.8):

  return librosa.effects.time_stretch(data, rate = rate)

def pitch(data, sr, pitch_factor=0.7):

  return librosa.effects.pitch_shift(data, sr=sr, n_steps=pitch_factor)

In [37]:
# loading the audio and extract features

all_f = []
all_l = []

for act_dir in tqdm(os.listdir(data_path)): # loop and do the following instructions
  act_path = os.path.join(data_path, act_dir)

  if not os.path.isdir(act_path): # for checking
    continue

  for file in os.listdir(act_path):

    if file.endswith(".wav"):
      # geting the label
      parts = file.split('-')

      label = emotion_map[parts[2]] # index 2 from the file name is the actual emotion

      file_path = os.path.join(act_path, file)

      data, sr = librosa.load(file_path, duration=3, offset=0.5) # load the file

      # extract from the original

      res1 = extract_features(data, sr)
      all_f.append(res1)
      all_l.append(label)

      # add noise and extract

      noise_data = noise(data)
      res2 = extract_features(noise_data, sr)
      all_f.append(res2)
      all_l.append(label)

      # pitch shift and extract

      pitch_data = pitch(data, sr)

      res3 = extract_features(pitch_data, sr)
      all_f.append(res3)
      all_l.append(label)

df = pd.DataFrame(
    all_f,
)
df['label'] = all_l

100%|██████████| 24/24 [14:24<00:00, 36.01s/it]


In [40]:
print(len(df))

4320


In [38]:
# now preparing and testing the data

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split


X = df.drop(
    columns=['label'] # droping the label so the model doesnot cheat.
).values

y = df['label'].values

le = LabelEncoder() # encoding object types to numbers
y_encoded = le.fit_transform(y)

scaler = StandardScaler() # scaling the numerical features
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42) # test size 0.2 so 0.8 for training

# reshaping so it's suitable for CNN

X_train = np.expand_dims(X_train, axis=2)
X_test = np.expand_dims(X_test, axis=2)

In [39]:
X_train.shape

(3456, 180, 1)

In [42]:
# now let's build the model and compile it

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.regularizers import l2


model = Sequential([
    Input(
        shape=(X_train.shape[1]-1, 1)
        ),
    Conv1D(
        256,
        8,
        padding='same',
        activation='relu',
        kernel_regularizer=l2(0.001)
    ),
    BatchNormalization(),

    MaxPooling1D(
        pool_size=5,
        strides=2,
        padding='same'
    ),

    Conv1D(
        128,
        8,
        padding='same',
        activation='relu',
        kernel_regularizer=l2(0.001)
    ),
    BatchNormalization(),

    MaxPooling1D(
        pool_size=5,
        strides=2,
        padding='same'
    ),

    Dropout(0.5),

    Flatten(),

    Dense(
        8,
        activation='softmax',
    )
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [43]:
from re import VERBOSE
# let's train it

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=500, # as we increase the epoch the accuracy will increase
    batch_size=32,
    verbose=1
)

Epoch 1/500
108/108 ━━━━━━━━━━━━━━━━━━━━ 27s 213ms/step - accuracy: 0.2541 - loss: 2.8054 - val_accuracy: 0.1944 - val_loss: 2.1136
Epoch 2/500
108/108 ━━━━━━━━━━━━━━━━━━━━ 42s 223ms/step - accuracy: 0.3229 - loss: 2.3475 - val_accuracy: 0.1910 - val_loss: 2.0616
Epoch 3/500
108/108 ━━━━━━━━━━━━━━━━━━━━ 25s 232ms/step - accuracy: 0.3568 - loss: 2.2073 - val_accuracy: 0.3056 - val_loss: 1.9277
Epoch 4/500
108/108 ━━━━━━━━━━━━━━━━━━━━ 38s 209ms/step - accuracy: 0.3773 - loss: 2.0914 - val_accuracy: 0.4248 - val_loss: 1.7587
Epoch 5/500
108/108 ━━━━━━━━━━━━━━━━━━━━ 25s 230ms/step - accuracy: 0.3993 - loss: 1.9842 - val_accuracy: 0.4907 - val_loss: 1.6316
Epoch 6/500
108/108 ━━━━━━━━━━━━━━━━━━━━ 41s 232ms/step - accuracy: 0.4172 - loss: 1.9270 - val_accuracy: 0.5023 - val_loss: 1.5747
Epoch 7/500
108/108 ━━━━━━━━━━━━━━━━━━━━ 22s 207ms/step - accuracy: 0.4306 - loss: 1.8301 - val_accuracy: 0.5127 - val_loss: 1.4801
Epoch 8/500
108/108 ━━━━━━━━━━━━━━━━━━━━ 41s 205ms/step - accuracy: 0.4343 -

In [53]:
# let's evaluate the model

test_loss, test_acc = model.evaluate(
    X_test,
    y_test
    )

print(f"\n Final Test Accuracy:"
f"{test_acc * 100:.2f}%")

27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.8090 - loss: 1.1606

 Final Test Accuracy:80.90%


In [54]:
# let's save the model, scaler, and Labelencoder

import joblib
from google.colab import files

model.save('emotion_recognition_from_speech.h5')

joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')

['label_encoder.pkl']

In [55]:
# downloading


files.download('emotion_recognition_from_speech.h5')
files.download('scaler.pkl')
files.download('label_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>